# Compare optimised pipeline runs (CSV diagnostics)

This notebook helps you compare multiple `optimised_processing` runs by:
- locating one-row diagnostics CSVs (recursively)
- concatenating them into a single table
- computing a few derived metrics (e.g. clearing %)
- plotting comparisons across `run_tag` / settings

The pipeline writes these diagnostics from the legacy method:
- `*_summary.csv` (one row per run)
- `*_sr_scale_verify.csv` (one row per run; band percentile audit)

If nothing is found, update `SEARCH_ROOTS` in Cell 2 to point at your run work directory.

# Terminal run prompts (EASI)

Copy/paste reference for running the NDVI build and three EDS A/B variants.

## Common variables (set once)

```bash
export EASI_REPO="/home/jovyan/work-easi-eds"
export EDS_BUCKET="dcceew-eds-data"
export EDS_PREFIX="AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds"

export NDVI_WORK="/home/jovyan/scratch/eds-work-optimised"
export EDS_WORK="/home/jovyan/scratch/eds-work-processing"

cd "$EASI_REPO"
```

## 1) Build NDVI products (required once per tile/date)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_ndvi/scripts/ndvi_master_pipeline.py" \
  --tile p089r080 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$NDVI_WORK" \
  --cloud-max 40 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09
```

## 2) Run EDS optimised processing (3 A/B variants)

All variants below write run-scoped outputs via `--run-id` and emit diagnostics CSVs via `--diagnostics`.

### Variant A: FORCE no SR scaling (debug baseline)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --legacy-no-auto-sr-scale \
  --run-id run-no-auto-sr-scale
```

### Variant B: AUTO SR scaling (recommended default)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --run-id run-auto-scale
```

### Variant C: MANUAL SR scaling (force 10000)

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --legacy-sr-scale 10000 \
  --run-id run-forced-10000
```

Tip: you generally don’t need `--rebase` for A/B runs if you keep unique `--run-id` values.

## Optional Variant D: legacy baseline includes nodata zeros

This adds `--legacy-baseline-include-nodata`, which makes the legacy baseline stats include nodata zeros in baseline mean/std/slope.
Default (recommended) behavior ignores zeros (treats 0 as nodata).

```bash
python "$EASI_REPO/scripts/easi-scripts/optimised_processing/scripts/eds_master_pipeline_optimised.py" \
  --tile p089r080 \
  --start-date 2025-06-07 \
  --end-date 2026-01-09 \
  --s3-bucket "$EDS_BUCKET" \
  --s3-prefix "$EDS_PREFIX" \
  --work-dir "$EDS_WORK" \
  --cloud-max 40 \
  --lookback 10 \
  --copy-to-home \
  --verbose \
  --diagnostics \
  --dlj-troubleshoot \
  --legacy-baseline-include-nodata \
  --run-id run-baseline-include-nodata
```

In [ ]:

from __future__ import annotations

from pathlib import Path
from io import BytesIO
import os
import re
import fnmatch

import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# S3 SEARCH CONFIG
# ============================================================

BUCKET = "dcceew-eds-data"
S3_ROOT = "EDS-work-testing/eds-work-processing"

SUMMARY_GLOB = "*_summary.csv"
SR_VERIFY_GLOB = "*_sr_scale_verify.csv"

# EASI should already provide access to this bucket/session.
s3 = boto3.client("s3", region_name="ap-southeast-2")


# ============================================================
# S3 SEARCH + READ HELPERS
# ============================================================

def list_s3_keys(bucket: str, prefix: str) -> list[str]:
    keys: list[str] = []
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=prefix.rstrip("/") + "/",
    ):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith("/"):
                keys.append(key)

    return keys


def find_s3_files(pattern: str) -> list[str]:
    keys = list_s3_keys(BUCKET, S3_ROOT)

    matches = [
        key for key in keys
        if fnmatch.fnmatch(Path(key).name, pattern)
    ]

    print(f"Searched: s3://{BUCKET}/{S3_ROOT}/")
    print(f"Pattern: {pattern}")
    print(f"Found: {len(matches)}")

    return matches


def read_s3_csv(key: str) -> pd.DataFrame:
    obj = s3.get_object(Bucket=BUCKET, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()))


def _infer_diag_suffix_from_key(key: str) -> str | None:
    name = Path(key).name
    stem = name

    for suffix in ("_summary.csv", "_sr_scale_verify.csv", ".csv"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
            break

    # Common diagnostic suffix pattern, for example:
    # sr-auto-10000_base-nodataaware
    m = re.search(r"(sr-[A-Za-z0-9-]+_base-[A-Za-z0-9-]+)$", stem)
    if m:
        return m.group(1)

    return None


def add_s3_path_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "s3_key" not in df.columns:
        print("[WARN] No s3_key column found.")
        return df

    def extract_tile(key: str):
        m = re.search(r"/(p\d{3}r\d{3})/", key)
        return m.group(1).lower() if m else None

    def extract_run_tag(key: str):
        m = re.search(r"/(p\d{3}r\d{3})/([^/]+)/", key)
        return m.group(2) if m else None

    df["tile"] = df["s3_key"].apply(extract_tile)
    df["run_tag"] = df["s3_key"].apply(extract_run_tag)
    df["diag_suffix"] = df["s3_key"].apply(_infer_diag_suffix_from_key)

    df["run_label"] = (
        df["run_tag"].fillna("unknown")
        + " | "
        + df["diag_suffix"].fillna("")
    ).str.strip(" |")

    return df


def read_many_s3_csvs(keys: list[str]) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    for key in keys:
        try:
            one = read_s3_csv(key)
            one["s3_key"] = key
            dfs.append(one)
        except Exception as e:
            print(f"[WARN] Failed to read {key}")
            print(e)

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


print("S3 helper cell loaded.")
print(f"S3 root: s3://{BUCKET}/{S3_ROOT}/")


In [ ]:
# S3 helpers are defined above.

In [ ]:
# Local SEARCH_ROOTS / LocatedFile helper cell removed for S3 version.

In [ ]:

# ============================================================
# Locate and load summary CSVs from S3
# ============================================================

summary_keys = find_s3_files(SUMMARY_GLOB)

print("Found summary CSVs:", len(summary_keys))

if summary_keys:
    print("First few:")
    for key in summary_keys[:5]:
        print(" -", key)

df_summary = read_many_s3_csvs(summary_keys)
df_summary = add_s3_path_fields(df_summary)

print("Summary table shape:", df_summary.shape)

df_summary.head(5)


In [ ]:

# If we found nothing, show a hint and stop early
if df_summary.empty:
    raise RuntimeError(
        "No *_summary.csv files found in S3.\n"
        f"Searched: s3://{BUCKET}/{S3_ROOT}/\n"
        "Check BUCKET and S3_ROOT in the S3 SEARCH CONFIG cell."
    )


In [ ]:

# ============================================================
# Derive comparison-friendly columns
# ============================================================

def _sum_cols(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    existing = [c for c in cols if c in df.columns]
    if not existing:
        return pd.Series([pd.NA] * len(df), index=df.index)
    return df[existing].fillna(0).sum(axis=1)


df = df_summary.copy()

# These are the main clearing classes used by the legacy method
clearing_classes = [34, 35, 36, 37, 38, 39]
clearing_cols = [f"dll_count_{c}" for c in clearing_classes]

df["dll_clearing_px"] = _sum_cols(df, clearing_cols)
df["dll_ndvi_only_px"] = df["dll_count_3"] if "dll_count_3" in df.columns else pd.NA
df["dll_no_clearing_px"] = df["dll_count_10"] if "dll_count_10" in df.columns else pd.NA

if "dll_total_px" in df.columns:
    df["dll_clearing_pct"] = (
        pd.to_numeric(df["dll_clearing_px"], errors="coerce")
        / pd.to_numeric(df["dll_total_px"], errors="coerce")
    ) * 100.0
else:
    df["dll_clearing_pct"] = pd.NA

# Ensure S3-derived path fields exist
df = add_s3_path_fields(df)

print("Comparison table shape:", df.shape)
df.head(5)


In [ ]:
# add_s3_path_fields() is defined in the first S3 helper cell.

In [ ]:
# df already includes S3 path fields and derived clearing columns.

In [ ]:
# Plot function is defined below.

### Interpretation of DLL clearing percentage by run

These plots compare the percentage of pixels classified as clearing (`DLL` classes 34–39) for multiple processing configurations across different Landsat tiles.

Each subplot represents a separate tile, while each bar represents a different workflow configuration combining:
- surface reflectance (SR) scaling behaviour, and
- baseline nodata handling.

The runs include:
- `sr-no-auto-1`: no SR scaling applied,
- `sr-manual-10000`: manually forcing SR scaling by 10000,
- `sr-auto-10000`: automatic SR scaling detection,
- `base-nodataaware`: nodata values excluded from baseline statistics,
- `base-legacy`: legacy behaviour where nodata zeros are included in the baseline.

Overall, the plots demonstrate that SR scaling has a major influence on clearing detection results.

Runs without SR scaling generally produce substantially larger clearing percentages, indicating that unscaled reflectance values inflate the spectral metrics and lead to excessive clearing detection. In contrast, runs using either automatic or manually forced scaling by 10000 produce more moderate and stable outputs.

The comparison between `sr-manual-10000` and `sr-auto-10000` shows very similar clearing percentages, suggesting that the automatic scaling detection is functioning correctly.

The `base-legacy` runs typically produce lower clearing percentages than the nodata-aware runs. This occurs because including nodata zeros in the historical baseline artificially increases baseline variability and suppresses anomaly detection, causing the workflow to under-detect clearing.

Differences between tiles indicate that landscape characteristics, vegetation structure, and seasonal conditions also influence clearing estimates. However, the plots clearly show that preprocessing decisions, particularly SR scaling and nodata handling, are major controls on workflow behaviour and output stability.

In [ ]:

# Plot 1: clearing % by run (one chart per tile)
def plot_clearing_by_tile(df: pd.DataFrame) -> None:
    required = ["tile", "run_tag", "diag_suffix", "run_label", "dll_clearing_pct"]
    missing = [c for c in required if c not in df.columns]

    if missing:
        print(f"[WARN] Missing columns: {missing}")
        print("Available columns:")
        print(list(df.columns))
        return

    tiles = [t for t in sorted(df["tile"].dropna().unique())]

    if not tiles:
        print("[WARN] No tiles found to plot.")
        return

    for tile in tiles:
        sub = df[df["tile"] == tile].copy()
        sub = sub.dropna(subset=["dll_clearing_pct"])

        if sub.empty:
            continue

        sub = sub.sort_values(["run_tag", "diag_suffix"], na_position="last")

        plt.figure(figsize=(10, max(3, 0.35 * len(sub))))
        y = np.arange(len(sub))

        plt.barh(y, sub["dll_clearing_pct"].astype(float))
        plt.yticks(y, sub["run_label"])
        plt.xlabel("DLL clearing pixels (% of total)")
        plt.title(f"{tile}: clearing % by run")
        plt.grid(axis="x", alpha=0.2)
        plt.tight_layout()
        plt.show()


plot_clearing_by_tile(df)


### Interpretation of clearing pixels vs SR scaling factor

This plot compares the number of pixels classified as clearing (`DLL` classes 34–39) against the surface reflectance (SR) scaling factor used in each run.

- Each point represents one processing run for a tile/date combination.
- The x-axis shows the SR scaling factor applied to the reflectance data.
- The y-axis shows the total number of pixels classified as clearing.

Two distinct groups are visible:

- **Scale factor near 0 or no scaling**: these runs generally produced much larger and more variable clearing outputs.
- **Scale factor = 10000**: these runs produced lower and more stable clearing outputs.

This indicates that the change detection workflow is highly sensitive to SR scaling. Without scaling, the spectral calculations operate on excessively large reflectance values (e.g. 0–10000 instead of 0–1), which inflates the spectral index and causes the algorithm to over-detect clearing.

Applying the expected SR scaling factor of 10000 stabilises the spectral metrics and reduces likely false positives, resulting in more physically realistic clearing estimates.

The spread of values within each group also shows that tile characteristics and seasonal conditions still influence the amount of detected clearing, but SR scaling is one of the dominant controls on output behaviour.

In [ ]:

# Plot 2: clearing pixels vs SR scaling factor (quick correlation check)
if "df" not in globals() or df.empty:
    print("[INFO] df has not been created yet.")
elif "dll_clearing_px" in df.columns and "sr_scale_factor" in df.columns:
    plt.figure(figsize=(8, 4))
    plt.scatter(
        df["sr_scale_factor"].astype(float),
        df["dll_clearing_px"].astype(float),
        alpha=0.7,
    )
    plt.xlabel("sr_scale_factor")
    plt.ylabel("dll_clearing_px (sum of classes 34..39)")
    plt.title("Clearing pixels vs SR scaling factor")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Needed columns not present for this plot.")
    print("Available columns:", list(df.columns) if "df" in globals() else [])


In [ ]:

# ============================================================
# Load SR scaling verification CSVs from S3
# ============================================================

sr_keys = find_s3_files(SR_VERIFY_GLOB)

print("Found SR verify CSVs:", len(sr_keys))

if sr_keys:
    print("First few:")
    for key in sr_keys[:5]:
        print(" -", key)

df_sr = read_many_s3_csvs(sr_keys)
df_sr = add_s3_path_fields(df_sr)

print("SR verify table shape:", df_sr.shape)

df_sr.head(3)


In [ ]:

# SR scaling audit plot: raw vs scaled median (p50) for start/end bands
def _maybe_col(df: pd.DataFrame, col: str) -> bool:
    return col in df.columns and df[col].notna().any()

if "df_sr" not in globals() or df_sr.empty:
    print("[INFO] No sr_scale_verify CSVs found (this is optional).")
else:
    dfp = df_sr.copy()
    dfp["run_label"] = (
        dfp.get("run_tag", pd.Series([""] * len(dfp))).fillna("")
        + " | "
        + dfp.get("diag_suffix", pd.Series([""] * len(dfp))).fillna("")
    ).str.strip(" |")

    possible_cols = [
        c for c in dfp.columns
        if ("p50" in c.lower() or "median" in c.lower())
        and pd.api.types.is_numeric_dtype(dfp[c])
    ]

    if not possible_cols:
        print("[INFO] No obvious median/p50 numeric columns found.")
        print("Available columns:", list(dfp.columns))
    else:
        for col in possible_cols[:6]:
            plt.figure(figsize=(10, max(3, 0.35 * len(dfp))))
            sub = dfp.sort_values(["tile", "run_tag", "diag_suffix"], na_position="last")
            y = np.arange(len(sub))
            plt.barh(y, sub[col].astype(float))
            plt.yticks(y, sub["run_label"])
            plt.xlabel(col)
            plt.title(f"SR scale verification: {col}")
            plt.grid(axis="x", alpha=0.2)
            plt.tight_layout()
            plt.show()


In [ ]:

# Save concatenated tables for later use
out_dir = Path.cwd().resolve() / "outputs" / "diagnostics_concat"
out_dir.mkdir(parents=True, exist_ok=True)

summary_out = out_dir / "all_runs_summary_concat.csv"
df.to_csv(summary_out, index=False)
print("Wrote:", summary_out)

if "df_sr" in globals() and not df_sr.empty:
    sr_out = out_dir / "all_runs_sr_scale_verify_concat.csv"
    df_sr.to_csv(sr_out, index=False)
    print("Wrote:", sr_out)
else:
    print("[INFO] No SR verify table to save.")
